## Monte Carlo Policy Evaluation

In [ ]:
import gymnasium as gym
import numpy as np
from collections import defaultdict
from tqdm.notebook import trange
import multiprocessing as mp

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Create Blackjack environment
env = gym.make('FrozenLake-v1', render_mode='human')

In [ ]:
def plot_frozenlake_env(filename='frozenlakeenv.png', greyscale=False):
    # Create the environment with render_mode=None
    env = gym.make('FrozenLake-v1', render_mode=None)
    env.reset()

    # Get the grid layout directly from the environment
    grid_array = env.unwrapped.desc.copy()

    # Create a mapping of characters to colors
    color_map = {
        b'S': [0.0, 1.0, 1.0],  # Cyan for Start
        b'F': [0.8, 0.8, 1.0],  # Light Blue for Frozen
        b'H': [1.0, 0.0, 0.0],  # Red for Hole
        b'G': [0.0, 1.0, 0.0]   # Green for Goal
    }
    if greyscale == True:
        color_map = {
            b'S': [0.587, 0.587, 0.587],  # Greyscale for Cyan
            b'F': [0.8471, 0.8471, 0.8471],  # Greyscale for Light Blue
            b'H': [0.299, 0.299, 0.299],  # Greyscale for Red
            b'G': [0.587, 0.587, 0.587]   # Greyscale for Green
        }


    # Create a 3D numpy array representing the RGB values of each cell
    rgb_array = np.array([[color_map[cell] for cell in row] for row in grid_array])

    # Create the plot
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(rgb_array)

    # Remove the axes ticks
    ax.set_xticks([])
    ax.set_yticks([])

    # Add grid lines
    ax.grid(which='major', axis='both', linestyle='-', color='k', linewidth=2)
    ax.set_xticks(np.arange(-.5, 4, 1), minor=True)
    ax.set_yticks(np.arange(-.5, 4, 1), minor=True)
    
    # Add text labels to each cell
    for i in range(4):
        for j in range(4):
            ax.text(j, i, grid_array[i, j].decode(), ha='center', va='center', fontsize=20, fontweight='bold')

    # Add a legend
    legend_elements = [
        plt.Rectangle((0, 0), 1, 1, facecolor=color_map[b'S'], edgecolor='none', label='Start'),
        plt.Rectangle((0, 0), 1, 1, facecolor=color_map[b'F'], edgecolor='none', label='Frozen'),
        plt.Rectangle((0, 0), 1, 1, facecolor=color_map[b'H'], edgecolor='none', label='Hole'),
        plt.Rectangle((0, 0), 1, 1, facecolor=color_map[b'G'], edgecolor='none', label='Goal')
    ]
    ax.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1, 0.5))

    plt.title('FrozenLake Environment')
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
# Call the function to plot the environment
plot_frozenlake_env()

In [ ]:
# Call the function to plot the environment
plot_frozenlake_env('frozenlakeenv_bw.png', True)

In [ ]:
def sample_policy(state):
    return env.action_space.sample()

In [ ]:
def generate_episode(env, policy):
    episode = []
    state, _ = env.reset()
    done = False
    while not done:
        action = policy(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        episode.append((state, action, reward))
        state = next_state
        done = terminated or truncated
    return episode

In [ ]:
def monte_carlo_policy_evaluation(env, policy, num_episodes=10000, first_visit=True):
    returns = defaultdict(list)
    value_function = defaultdict(float)

    for _ in trange(num_episodes):
        episode = generate_episode(env, policy)
        state_action_pairs = set()
        for t, (state, action, reward) in enumerate(episode):
            if first_visit and (state, action) in state_action_pairs:
                continue
            state_action_pairs.add((state, action))
            G = sum(r for _, _, r in episode[t:])
            returns[(state, action)].append(G)
            value_function[(state, action)] = np.mean(returns[(state, action)])

    return value_function

In [ ]:
first_visit_value_function = monte_carlo_policy_evaluation(env, sample_policy, 1000)

In [ ]:
every_visit_value_function = monte_carlo_policy_evaluation(env, sample_policy, 1000, False)

In [ ]:
def plot_value_function(value_function, env, title, filename, cmap='viridis'):
    # Create a 4x4 grid for the FrozenLake
    v = np.zeros((4, 4))
    
    # Fill the grid with the maximum value for each state
    for state in range(16):
        row, col = state // 4, state % 4
        v[row, col] = max([value_function.get((state, action), 0) for action in range(4)])
    
    # Create a figure and axis
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create a heatmap
    im = ax.imshow(v, cmap=cmap)
    
    # Add colorbar
    cbar = ax.figure.colorbar(im, ax=ax)
    cbar.ax.set_ylabel('Value', rotation=-90, va="bottom")
    
    # Add text annotations to the cells
    textcolor = 'w'
    if cmap == "Greys":
        textcolor = '0.5'
    for i in range(4):
        for j in range(4):
            text = ax.text(j, i, f"{v[i, j]:.2f}", 
                           ha="center", va="center", color=textcolor)
    
    # Customize the plot
    ax.set_title(title)
    ax.set_xticks(np.arange(4))
    ax.set_yticks(np.arange(4))
    ax.set_xticklabels(['0', '1', '2', '3'])
    ax.set_yticklabels(['0', '1', '2', '3'])
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    
    # Add grid lines
    ax.set_xticks(np.arange(-.5, 4, 1), minor=True)
    ax.set_yticks(np.arange(-.5, 4, 1), minor=True)
    ax.grid(which="minor", color="w", linestyle='-', linewidth=2)
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
plot_value_function(first_visit_value_function, env, "First-Visit MC Value Function", "FirstVisitVF.png")

In [ ]:
plot_value_function(first_visit_value_function, env, "First-Visit MC Value Function", "FirstVisitVF_bw.png", "Greys")

In [ ]:
plot_value_function(every_visit_value_function, env, "Every-Visit MC Value Function", "EveryVisitVF.png")

In [ ]:
plot_value_function(every_visit_value_function, env, "Every-Visit MC Value Function", "EveryVisitVF_bw.png", "Greys")